# Isolation Forest to predict customer churn
The goal here is to predict if the customer is likely to churn. 

Data collected from Kaggle: https://www.kaggle.com/code/use9009/credit-card-customers-predicting-churn

An Isolation Forest works by detecting if the data has a short path to single it out. This means if the data is very different, it can be separated quickly from the rest of the data, which will make it an anomaly (here it means it is likely to churn).

Isolation Forests are used in anomaly detection, so I will try to make another project based on that.


### Import libraries and packages

In [195]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score, classification_report

import plotly.express as px
import plotly.graph_objects as go

### Data wrangling/cleaning

In [196]:
df = pd.read_csv("data/BankChurners.csv")
df = df.iloc[:, :-2]
df['Attrition'] = df['Attrition_Flag'].apply(lambda x: 1 if x=='Attrited Customer' else 0)

df['Male'] = df['Gender'].apply(lambda x: 1 if x=='M' else 0)

df['edu'] = df['Education_Level'].map({'Unknown':0, 'Uneducated':1, 'High School':2, 'College':3, 'Graduate':4, 'Post-Graduate':5, 'Doctorate':6}) # not sure if i should replace unknown with nan

df['Married'] = df['Marital_Status'].map({'Unknown':-1, 'Single':0, 'Divorced':1, 'Married': 2})

df['Income'] = df['Income_Category'].map({'Unknown':0, 'Less than $40K':1, '$40K - $60K':2, '$60K - $80K':3, '$80K - $120K':4, '$120K +':5})

df['Card'] = df['Card_Category'].map({'Blue':1, 'Silver':2, 'Gold':3, 'Platinum':4})

df.drop(columns=['Attrition_Flag', 'Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category'], inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CLIENTNUM                 10127 non-null  int64  
 1   Customer_Age              10127 non-null  int64  
 2   Dependent_count           10127 non-null  int64  
 3   Months_on_book            10127 non-null  int64  
 4   Total_Relationship_Count  10127 non-null  int64  
 5   Months_Inactive_12_mon    10127 non-null  int64  
 6   Contacts_Count_12_mon     10127 non-null  int64  
 7   Credit_Limit              10127 non-null  float64
 8   Total_Revolving_Bal       10127 non-null  int64  
 9   Avg_Open_To_Buy           10127 non-null  float64
 10  Total_Amt_Chng_Q4_Q1      10127 non-null  float64
 11  Total_Trans_Amt           10127 non-null  int64  
 12  Total_Trans_Ct            10127 non-null  int64  
 13  Total_Ct_Chng_Q4_Q1       10127 non-null  float64
 14  Avg_Ut

### Data splitting/scaling

In [216]:
X = df.drop(columns=['CLIENTNUM', 'Attrition'])
y = df['Attrition']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0, stratify=y)

### Isolation Forest model training

In [221]:
iso_forest = IsolationForest(n_estimators=25, max_samples='auto', contamination=0.1, random_state=0)
iso_forest.fit(X_train)

,n_estimators,25
,max_samples,'auto'
,contamination,0.1
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,0
,verbose,0
,warm_start,False


### Model predictions and evaluations

In [222]:
# Predictions: -1 for outliers, 1 for inliers
y_pred = iso_forest.predict(X_test)
# 1 for attrited, 0 for not attrited
y_pred_mapped = np.where(y_pred == -1, 1, 0)

classif_report = classification_report(y_test, y_pred_mapped)
acc = accuracy_score(y_test, y_pred_mapped)
print(f"Accuracy: {100*acc:.4f}%")
print(classif_report)

Accuracy: 78.5291%
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1701
           1       0.22      0.14      0.17       325

    accuracy                           0.79      2026
   macro avg       0.53      0.52      0.52      2026
weighted avg       0.75      0.79      0.76      2026



### ROC-AUC Curve

In [223]:
fpr, tpr, _ = roc_curve(y_test, iso_forest.decision_function(X_test))
roc_auc = roc_auc_score(y_test, iso_forest.decision_function(X_test))
print(f"ROC-AUC Score: {roc_auc:.4f}")

fig = go.Figure()

fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='model'))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash')))

fig.update_layout(
    title='ROC Curve',
    xaxis_title='False Positive Rate (FPR)',
    yaxis_title='True Positive Rate (TPR)',
    width=600, height=600
)
fig.show()

ROC-AUC Score: 0.4567


Why's this curve less than 0.5???

I was gonna do one that allows user input, but if you have to input all of that... eh, too much work right.

Well, this one was sadly the one I expected the most of but got pretty much the worst results for...
- The accuracy isn't very good
- Despite there being 10k lines and around 20 different attributes, I am dissapointed the accuracy and even the auc was not very high.
- There wasn't much I thought about expanding on this

In the future (in about a month), I will make another repository of these 4 models again but to a different usage; I will study and research more to understand more models and become a better data scientist! This is just one part of my journey, and hopefully I will learn so much more in the future :)



In [224]:
df.info

<bound method DataFrame.info of        CLIENTNUM  Customer_Age  Dependent_count  Months_on_book  \
0      768805383            45                3              39   
1      818770008            49                5              44   
2      713982108            51                3              36   
3      769911858            40                4              34   
4      709106358            40                3              21   
...          ...           ...              ...             ...   
10122  772366833            50                2              40   
10123  710638233            41                2              25   
10124  716506083            44                1              36   
10125  717406983            30                2              36   
10126  714337233            43                2              25   

       Total_Relationship_Count  Months_Inactive_12_mon  \
0                             5                       1   
1                             6              